## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting? 
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Emma Chung

**Date:** 9/30/2026

Q1.
1. Regression and classification are both methods used to make predictions, but they predict different types of outcomes. Regression predicts a numerical or continuous value while classification predicts a category or class. 
2. A confusion table compares the actual class of each observation with the class predicted by the model. Correct predictions appear along the diagonal of the table, while incorrect predictions appear in the other cells. It helps us see not only how often the model is correct, but also which classes the model tends to confuse with one another. This can provide more information about model performance than overall accuracy alone.
3. SSE stands for Sum of Squared Errors. It measures the total squared difference between the actual values and the values predicted by a model. A smaller SSE means that the model's predictions are generally closer to the actual values. Because the errors are squared, larger prediction errors contribute more heavily to the SSE.
4. Overfitting occurs when a model fits the training data too closely and begins capturing random noise or patterns that do not generalize to new data. Underfitting occurs when a model is too simple to capture the important patterns in the data. 
5. Splitting the data into training and testing sets lets us train the model on one set of data and see how well it works on new data. We can try different values of k and choose the one that gives the best accuracy or lowest SSE on the test set. This helps us choose a model that performs well on data it has not seen before.
6. A class label gives one clear prediction, which makes it easy to understand. However, it does not show how confident the model is in that prediction. A probability distribution gives the probability of each possible class, so it shows the model's uncertainty. However, it can be more difficult to interpret than a single class label.

Q2.

In [2]:
import pandas as pd

df = pd.read_csv("data/land_mines.csv")

df.head()

,voltage,height,soil,mine_type
0,0.338157,0.000000,0.0,1
1,0.320241,0.181818,0.0,1
2,0.287009,0.272727,0.0,1
3,0.256284,0.454545,0.0,1
4,0.262840,0.545455,0.0,1


In [3]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB


,voltage,height,soil,mine_type
count,338.000000,338.000000,338.000000,338.000000
mean,0.430634,0.508876,0.503550,2.952663
std,0.195819,0.306043,0.344244,1.419703
min,0.197734,0.000000,0.000000,1.000000
25%,0.309737,0.272727,0.200000,2.000000
50%,0.359516,0.545455,0.600000,3.000000
75%,0.482628,0.727273,0.800000,4.000000
max,0.999999,1.000000,1.000000,5.000000


In [4]:
df["mine_type"].value_counts().sort_index()

mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64

In [5]:
df.groupby("mine_type")[["voltage", "height", "soil"]].mean()

,voltage,height,soil
mine_type,,,
1,0.296463,0.495519,0.492958
2,0.721123,0.487013,0.508571
3,0.402408,0.527548,0.496970
4,0.345365,0.506887,0.503030
5,0.379598,0.530070,0.516923


In [6]:
df.isnull().sum()

voltage      0
height       0
soil         0
mine_type    0
dtype: int64

1. The dataset contains 338 observations. The three features are voltage, height, and soil, while mine_type is the target variable with classes ranging from 1 to 5. Voltage ranges from about 0.198 to 1, while height and soil both range from 0 to 1. There are no missing values in these columns. The mine types are also fairly evenly distributed, with 71 observations of Type 1, 70 of Type 2, 66 each of Types 3 and 4, and 65 of Type 5.

In [7]:
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]
y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.5,
    random_state=42,
    stratify=y
)

print("Training set:", len(X_train))
print("Test set:", len(X_test))

Training set: 169
Test set: 169


2. I split the data evenly into training and testing sets, using 50% of the observations for each. I stratified by mine type so that the different mine classes remained approximately evenly represented in both sets.

In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [9]:
accuracies = []

for k in range(1, 21):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    
    y_pred = knn.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    accuracies.append(accuracy)
    print(f"k = {k}: Accuracy = {accuracy:.3f}")

k = 1: Accuracy = 0.402
k = 2: Accuracy = 0.438
k = 3: Accuracy = 0.379
k = 4: Accuracy = 0.355
k = 5: Accuracy = 0.314
k = 6: Accuracy = 0.325
k = 7: Accuracy = 0.314
k = 8: Accuracy = 0.308
k = 9: Accuracy = 0.349
k = 10: Accuracy = 0.337
k = 11: Accuracy = 0.355
k = 12: Accuracy = 0.337
k = 13: Accuracy = 0.349
k = 14: Accuracy = 0.314
k = 15: Accuracy = 0.331
k = 16: Accuracy = 0.343
k = 17: Accuracy = 0.361
k = 18: Accuracy = 0.337
k = 19: Accuracy = 0.337
k = 20: Accuracy = 0.343


In [10]:
best_k = accuracies.index(max(accuracies)) + 1

print("Best k:", best_k)
print("Best accuracy:", max(accuracies))

Best k: 2
Best accuracy: 0.4378698224852071


3. I built a k-nearest neighbors classifier and tested values of k from 1 to 20. For each value, I trained the model on the training data and calculated its accuracy on the test data. I selected k = 2 because it had the highest test accuracy of about 43.8%.

In [11]:
knn = KNeighborsClassifier(n_neighbors=2)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

In [12]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

confusion_table = pd.DataFrame(
    cm,
    index=["Actual 1", "Actual 2", "Actual 3", "Actual 4", "Actual 5"],
    columns=["Predicted 1", "Predicted 2", "Predicted 3", "Predicted 4", "Predicted 5"]
)

confusion_table

,Predicted 1,Predicted 2,Predicted 3,Predicted 4,Predicted 5
Actual 1,25,0,6,4,1
Actual 2,0,32,0,3,0
Actual 3,12,2,9,6,4
Actual 4,12,5,8,8,0
Actual 5,15,2,10,5,0


4. The optimal model had an overall accuracy of about 43.8%. The model performed best for mine types 1 and 2. It correctly classified 25 of 36 Type 1 mines and 32 of 35 Type 2 mines. Performance was worse for Types 3 and 4, with only 9 of 33 Type 3 mines and 8 of 33 Type 4 mines correctly classified. The model performed worst for Type 5, where none of the 32 observations were correctly classified. Type 5 mines were most often incorrectly predicted as Type 1 or Type 3.

5. I would not recommend using this model by itself to identify a mine because it makes a large number of classification errors. In particular, it fails to correctly identify any Type 5 mines in the test set. The model could be used as an additional source of information, but its predictions should be confirmed using other methods before someone makes a decision. This is especially important in a situation involving land mines, where an incorrect prediction could have serious consequences.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]